System Path Setup

In [20]:
# Cell 1: Robust System Path Setup for Project Modules
import os
import sys

notebook_path = os.getcwd() # This should be 'gaias_ark_mangroves/notebooks/'
project_root = os.path.abspath(os.path.join(notebook_path, os.pardir))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root added to sys.path: {project_root}")
print(f"'configs' folder exists at root: {os.path.isdir(os.path.join(project_root, 'configs'))}")
print(f"'regions.py' file exists: {os.path.isfile(os.path.join(project_root, 'configs', 'regions.py'))}")

Project root added to sys.path: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark
'configs' folder exists at root: True
'regions.py' file exists: True


Import Core Libraries and GEE Initialization 

In [21]:
# Cell 2: Import Core Libraries and Initialize GEE
import ee
import pandas as pd
import geopandas as gpd
import folium # For interactive mapping
from pygbif import occurrences # For GBIF data
from configs.regions import kenyan_coast_roi # Your ROI

# Initialize GEE (essential for all GEE operations)
ee.Initialize(project='gaias-ark') # <--- REPLACE 'gaias-ark' with YOUR GEE Project ID

print("All core libraries imported and GEE initialized.")

All core libraries imported and GEE initialized.


Load Cleaned GBIF Data and Mangrove Extent

In [22]:
# Cell 3: Load Cleaned GBIF Data and GEE Mangrove Extent (REVISED: Define mangroves_extent_gee for Cell 5)
print("--- Loading Cleaned GBIF Data and Mangrove Extent ---")

# Load the cleaned GBIF GeoJSON locally
gbif_cleaned_path = os.path.join(project_root, 'data', 'processed', 'gbif_kenya_coastal_occurrences_cleaned.geojson')
if os.path.exists(gbif_cleaned_path):
    gbif_gdf_local = gpd.read_file(gbif_cleaned_path)
    print(f"Loaded {len(gbif_gdf_local)} cleaned GBIF records locally.")
else:
    print(f"Error: Cleaned GBIF data not found at {gbif_cleaned_path}. Please re-run 04_Species_GBIF_Acquisition.ipynb.")
    sys.exit("No cleaned GBIF data to process.")

essential_local_cols = [
    'gbifID', 'scientificName', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus',
    'species', 'eventDate', 'decimalLatitude', 'decimalLongitude'
]
gbif_gdf_clean_for_gee = gbif_gdf_local[essential_local_cols + ['geometry']].dropna(subset=['decimalLatitude', 'decimalLongitude'])

features_list = []
for index, row in gbif_gdf_clean_for_gee.iterrows():
    properties = {col: str(row[col]) if pd.api.types.is_object_dtype(type(row[col])) else row[col] for col in essential_local_cols}
    ee_geometry = ee.Geometry.Point([row.geometry.x, row.geometry.y])
    features_list.append(ee.Feature(ee_geometry, properties))
gbif_fc_gee = ee.FeatureCollection(features_list)
print(f"Converted {gbif_fc_gee.size().getInfo()} (cleaned) GBIF records to GEE FeatureCollection with essential properties only.")

# --- Load GEE Mangrove Extent image and define mangroves_extent_gee ---
mangrove_collection = ee.ImageCollection("LANDSAT/MANGROVE_FORESTS")
recent_mangrove_image = mangrove_collection \
    .filterBounds(kenyan_coast_roi) \
    .sort('system:time_start', False) \
    .first()
if not recent_mangrove_image:
    raise Exception("No recent mangrove images found for the ROI in LANDSAT/MANGROVE_FORESTS collection.")

# Define mangroves_extent_gee here for use in Cell 5 and beyond
# This will be the UNBUFFERED mangrove extent for visualization context
mangroves_extent_gee = recent_mangrove_image.clip(kenyan_coast_roi).gt(0).unmask(0) # <--- DEFINITION MOVED/CONFIRMED HERE!

print("GEE Mangrove Extent image loaded (for context).")

--- Loading Cleaned GBIF Data and Mangrove Extent ---


Loaded 209 cleaned GBIF records locally.
Converted 209 (cleaned) GBIF records to GEE FeatureCollection with essential properties only.
GEE Mangrove Extent image loaded (for context).


 Spatially Filter GBIF Points to Mangrove Areas

In [23]:
# Cell 4: Spatially Filter GBIF Points by Sampling Mangrove Raster (FULL REVISED METHOD WITH BUFFER)
print("--- Spatially Filtering GBIF data to Mangrove Extent (Alternative Method with Buffer) ---")

# --- Re-load cleaned GBIF data and convert to GEE FeatureCollection ---
# This block is duplicated from Cell 3 to make Cell 4 runnable independently after kernel restart
# In a production script, you'd pass gbif_fc_gee from Cell 3 or save as GEE Asset.
gbif_cleaned_path = os.path.join(project_root, 'data', 'processed', 'gbif_kenya_coastal_occurrences_cleaned.geojson')
if os.path.exists(gbif_cleaned_path):
    gbif_gdf_local = gpd.read_file(gbif_cleaned_path)
    print(f"Loaded {len(gbif_gdf_local)} cleaned GBIF records locally.")
else:
    print(f"Error: Cleaned GBIF data not found at {gbif_cleaned_path}. Please re-run 04_Species_GBIF_Acquisition.ipynb.")
    sys.exit("No cleaned GBIF data to process.")

essential_local_cols = [
    'gbifID', 'scientificName', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus',
    'species', 'eventDate', 'decimalLatitude', 'decimalLongitude'
]
gbif_gdf_clean_for_gee = gbif_gdf_local[essential_local_cols + ['geometry']].dropna(subset=['decimalLatitude', 'decimalLongitude'])

features_list = []
for index, row in gbif_gdf_clean_for_gee.iterrows():
    properties = {col: str(row[col]) if pd.api.types.is_object_dtype(type(row[col])) else row[col] for col in essential_local_cols}
    ee_geometry = ee.Geometry.Point([row.geometry.x, row.geometry.y])
    features_list.append(ee.Feature(ee_geometry, properties))
gbif_fc_gee = ee.FeatureCollection(features_list)
print(f"Converted {gbif_fc_gee.size().getInfo()} (cleaned) GBIF records to GEE FeatureCollection with essential properties only.")

# --- Load GEE Mangrove Extent image ---
mangrove_collection = ee.ImageCollection("LANDSAT/MANGROVE_FORESTS")
recent_mangrove_image = mangrove_collection \
    .filterBounds(kenyan_coast_roi) \
    .sort('system:time_start', False) \
    .first()
if not recent_mangrove_image:
    raise Exception("No recent mangrove images found for the ROI in LANDSAT/MANGROVE_FORESTS collection.")

# Ensure the mangrove image is binary (1 for mangrove, 0 for non-mangrove) and unmask 0s explicitly.
# We rename the band to 'is_mangrove' for clarity in the sampled properties.
mangroves_raster_mask = recent_mangrove_image.clip(kenyan_coast_roi).gt(0).unmask(0).rename('is_mangrove')

# --- REVISED: Significantly larger buffer around the mangrove mask ---
# Let's try 300 meters (10 pixels at 30m resolution) to ensure overlap
buffered_mangroves_raster_mask = mangroves_raster_mask.focal_max(10).reproject(crs=mangroves_raster_mask.projection().crs(), scale=30) # <--- Focal max of 10
buffered_mangroves_raster_mask = buffered_mangroves_raster_mask.updateMask(buffered_mangroves_raster_mask.gt(0)).rename('is_mangrove')

print("GEE Mangrove Extent raster mask loaded, buffered (300m), and prepared for sampling.")


# --- Sample the 'is_mangrove' band from the *buffered* mask ---
# This adds a new property to each GBIF feature with the value of the 'is_mangrove' band at that point.
sampled_gbif_fc = buffered_mangroves_raster_mask.reduceRegions(
    collection=gbif_fc_gee,
    reducer=ee.Reducer.first(), # Get the value of the 'is_mangrove' band at each point's location
    scale=30, # Match the resolution of the mangrove image
    crs=buffered_mangroves_raster_mask.projection().crs()
)

# Filter the sampled points: keep only those where 'is_mangrove' (the output of reducer.first()) is 1.
gbif_in_mangroves_gee = sampled_gbif_fc.filter(ee.Filter.eq('first', 1))

# Rename 'first' to 'is_mangrove' (server-side)
def rename_first_to_is_mangrove(feature):
    # Use .set() to add 'is_mangrove' and .set('first', None) to effectively remove 'first'
    return feature.set('is_mangrove', feature.get('first')).set('first', None)

gbif_final_selection_gee = gbif_in_mangroves_gee.map(rename_first_to_is_mangrove)

# --- Initiate GEE Export Task ---
final_export_properties = essential_local_cols + ['is_mangrove']

# Add a check for the size BEFORE initiating export
final_collection_size = gbif_final_selection_gee.size().getInfo()
print(f"Number of GBIF records in final FeatureCollection before export: {final_collection_size}")

if final_collection_size == 0:
    print("WARNING: The final FeatureCollection is empty after filtering. Export task will be skipped.")
    print("Consider adjusting ROI or buffer size if this is unexpected.")
    gbif_in_mangroves_local = gpd.GeoDataFrame() # Assign empty DF to prevent subsequent errors
else:
    my_gee_project_id = 'gaias-ark'
    output_asset_id_prefix = f'projects/{my_gee_project_id}/assets/gaias_ark_gbif_mangrove_occurrences'
    output_description = 'GBIF Mangrove Occurrences Kenya'

    task = ee.batch.Export.table.toAsset(
        collection=gbif_final_selection_gee.select(final_export_properties),
        description=output_description,
        assetId=output_asset_id_prefix,
        selectors=final_export_properties
    )

    task.start()
    print(f"\nGEE Export Task initiated for spatially filtered GBIF data. Asset ID: {output_asset_id_prefix}")
    print("Check the 'Tasks' tab in your GEE Code Editor to monitor progress.")
    print("\nOnce the GEE task completes, you will need to load the exported asset (FeatureCollection) in a new cell:")
    print(f"   # Example to load: ee.FeatureCollection('{output_asset_id_prefix}')")

    gbif_in_mangroves_local = gpd.GeoDataFrame() # Assign empty DF for local continuation

--- Spatially Filtering GBIF data to Mangrove Extent (Alternative Method with Buffer) ---
Loaded 209 cleaned GBIF records locally.
Converted 209 (cleaned) GBIF records to GEE FeatureCollection with essential properties only.
GEE Mangrove Extent raster mask loaded, buffered (300m), and prepared for sampling.
Number of GBIF records in final FeatureCollection before export: 1

GEE Export Task initiated for spatially filtered GBIF data. Asset ID: projects/gaias-ark/assets/gaias_ark_gbif_mangrove_occurrences
Check the 'Tasks' tab in your GEE Code Editor to monitor progress.

Once the GEE task completes, you will need to load the exported asset (FeatureCollection) in a new cell:
   # Example to load: ee.FeatureCollection('projects/gaias-ark/assets/gaias_ark_gbif_mangrove_occurrences')


Visualize Spatially Filtered Data

In [24]:
# Cell 5 (REVISED AGAIN): Load Exported GBIF Asset and Visualize (Ensuring Display)
print("--- Loading Exported GBIF Asset and Visualizing ---")

exported_gbif_asset_id = 'projects/gaias-ark/assets/gaias_ark_gbif_mangrove_occurrences'

gbif_in_mangroves_asset = ee.FeatureCollection(exported_gbif_asset_id)

try:
    gbif_in_mangroves_local = gpd.GeoDataFrame.from_features(gbif_in_mangroves_asset.getInfo()['features'], crs='EPSG:4326')
    print(f"Successfully loaded {len(gbif_in_mangroves_local)} spatially filtered GBIF records from GEE Asset.")

    if not gbif_in_mangroves_local.empty:
        centroid_coords = kenyan_coast_roi.centroid().getInfo()['coordinates']
        center_lat, center_lon = centroid_coords[1], centroid_coords[0]

        m_filtered = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles='OpenStreetMap')

        folium.GeoJson(
            kenyan_coast_roi.getInfo(),
            name='Kenyan Coastal ROI',
            style_function=lambda x: {'fillColor': '#f08080', 'color': 'red', 'weight': 3, 'fillOpacity': 0.6}
        ).add_to(m_filtered)

        mangrove_vis_params = {'min': 0, 'max': 1, 'palette': ['white', 'green']}
        map_id_dict_mangroves = mangroves_extent_gee.getMapId(mangrove_vis_params)
        folium.TileLayer(
            tiles=map_id_dict_mangroves['tile_fetcher'].url_format,
            attr='Google Earth Engine',
            overlay=True,
            name='Mangrove Extent'
        ).add_to(m_filtered)

        folium.GeoJson(
            gbif_in_mangroves_local.__geo_interface__,
            name='GBIF in Mangroves',
            tooltip=folium.features.GeoJsonTooltip(fields=['scientificName'], aliases=['Species']),
            marker=folium.CircleMarker(radius=8, weight=2, color='blue', fill_color='lightblue', fill_opacity=0.7)
        ).add_to(m_filtered)

        folium.LayerControl().add_to(m_filtered)
        
        # --- CRITICAL CHANGE: Ensure map object is the last expression or explicitly displayed ---
        m_filtered # <--- Add this line if it's not already the very last thing.
        # Alternatively, if you have other prints/code after, use:
        # from IPython.display import display
        # display(m_filtered)

    else:
        print("No GBIF records found in the asset to visualize.")

except Exception as e:
    print(f"Error loading exported GBIF asset or visualizing: {e}")
    print("Please ensure the GEE export task completed successfully in the GEE Code Editor.")
    gbif_in_mangroves_local = gpd.GeoDataFrame()

--- Loading Exported GBIF Asset and Visualizing ---
Successfully loaded 1 spatially filtered GBIF records from GEE Asset.


In [29]:
# Cell 6: Extract Environmental Variables for Filtered GBIF Points (FULL REVISED WITH ROBUST GEOMETRY)
print("--- Extracting Environmental Variables for GBIF Occurrences ---")

# --- Re-load necessary GEE data and assets for Cell 6 independence ---
# (These blocks are duplicated to make Cell 6 runnable independently after kernel restart)

# Re-initialize GEE if running this cell alone (already done in Cell 2)
# ee.Initialize(project='gaias-ark') # Ensure project ID is correct

# Load ROI (from config)
# from config.regions import kenyan_coast_roi # Assuming imports from Cell 2 are active

# Load environmental data
nasadem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')
elevation_roi = nasadem.clip(kenyan_coast_roi)

worldclim_dataset = ee.ImageCollection("WORLDCLIM/V1/MONTHLY")
annual_mean_temp = worldclim_dataset.select('tavg').mean().divide(10).rename('mean_annual_temp_C')
annual_total_prec = worldclim_dataset.select('prec').sum().rename('total_annual_prec_mm')
mean_temp_roi = annual_mean_temp.clip(kenyan_coast_roi)
total_prec_roi = annual_total_prec.clip(kenyan_coast_roi)

environmental_stack = ee.Image.cat([
    elevation_roi.rename('elevation_m'),
    mean_temp_roi,
    total_prec_roi
])
print("Environmental data (Elevation, Temp, Precip) loaded for extraction.")

# Load the spatially filtered GBIF FeatureCollection from GEE Asset (exported from Cell 4)
exported_gbif_asset_id = 'projects/gaias-ark/assets/gaias_ark_gbif_mangrove_occurrences' # <--- CONFIRM YOUR GEE PROJECT ID
gbif_in_mangroves_asset = ee.FeatureCollection(exported_gbif_asset_id)


num_records_in_asset = gbif_in_mangroves_asset.size().getInfo()
print(f"Loaded {num_records_in_asset} GBIF records from asset for environmental extraction.")

if num_records_in_asset > 0:
    # Define a list of *expected* original GBIF properties we want to keep.
    properties_to_keep = [
        'gbifID', 'scientificName', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus',
        'species', 'eventDate', 'decimalLatitude', 'decimalLongitude', 'is_mangrove'
    ]

    def sample_env_for_feature(feature):
        # Sample the environmental stack at the feature's geometry.
        # This returns a single ee.Feature where properties are the sampled values.
        sampled_feature_with_env_props = environmental_stack.sample(
            region=feature.geometry(),
            scale=30,
            dropNulls=False
        ).first()

        # Merge the original feature's selected properties with the sampled environmental properties.
        # Use feature.set(sampled_feature_with_env_props.propertyNames().reduce(ee.Reducer.toList()),
        # sampled_feature_with_env_props.values()) -- this is too complex.
        
        # The most straightforward way to combine properties is to use `copyProperties`.
        # First, filter the original feature to only desired properties.
        clean_original_feature = feature.select(properties_to_keep)
        
        # Then, copy the environmental properties from the sampled feature into the cleaned original feature.
        # .copyProperties(source, [properties to copy])
        # The sampled_feature_with_env_props has bands like 'elevation_m', 'mean_annual_temp_C', etc.
        
        # Get the names of the environmental bands
        env_band_names = environmental_stack.bandNames()
        
        # Copy all properties (which are the sampled environmental values) from the sampled_feature
        # onto the original feature, while preserving its geometry and original selected properties.
        return clean_original_feature.copyProperties(sampled_feature_with_env_props, env_band_names)
        

    # Map this function over the FeatureCollection to enrich each point.
    gbif_with_env_gee = gbif_in_mangroves_asset.map(sample_env_for_feature)
    print("Environmental variables extracted to GBIF features.")

    # Convert the GEE FeatureCollection with environmental data to a local GeoDataFrame
    try:
        gbif_with_env_local = gpd.GeoDataFrame.from_features(
            gbif_with_env_gee.getInfo()['features'],
            crs='EPSG:4326'
        )
        print(f"Successfully downloaded {len(gbif_with_env_local)} GBIF records with environmental data.")

        output_env_path = os.path.join(project_root, 'data', 'processed', 'gbif_kenya_mangrove_env_enriched.geojson')
        gbif_with_env_local.to_file(output_env_path, driver='GeoJSON')
        print(f"GBIF data enriched with environmental variables saved to: {output_env_path}")

        print("\nHead of GBIF data with environmental variables:")
        print(gbif_with_env_local.head())

    except Exception as e:
        print(f"Error downloading GBIF data with environmental variables: {e}")
        print("This might happen if the result is still too large or has other download issues.")
        gbif_with_env_local = gpd.GeoDataFrame()
else:
    print("No GBIF records in asset to enrich with environmental data; skipping extraction.")
    gbif_with_env_local = gpd.GeoDataFrame()

--- Extracting Environmental Variables for GBIF Occurrences ---
Environmental data (Elevation, Temp, Precip) loaded for extraction.
Loaded 1 GBIF records from asset for environmental extraction.
Environmental variables extracted to GBIF features.


INFO:Created 1 records


Successfully downloaded 1 GBIF records with environmental data.
GBIF data enriched with environmental variables saved to: c:\Users\pickle-ian\OneDrive\Desktop\Gaia\Gaia's Ark\data\processed\gbif_kenya_mangrove_env_enriched.geojson

Head of GBIF data with environmental variables:
                   geometry       class  decimalLatitude  decimalLongitude  \
0  POINT (39.40511 -4.6449)  Asteroidea        -4.644905         39.405113   

   elevation_m            eventDate           family      gbifID    genus  \
0            0  2025-01-26 07:09:56  Ophidiasteridae  5063468935  Linckia   

   is_mangrove   kingdom      order         phylum  \
0            1  Animalia  Valvatida  Echinodermata   

                      scientificName            species  
0  Linckia multifora (Lamarck, 1816)  Linckia multifora  


In [31]:
# Cell 7: Load Exported AGC Density Asset and Consolidate for Analysis
print("--- Consolidating Data for Carbon Analysis ---")

# --- Load the Exported AGC Density Asset ---
my_gee_project_id = 'gaias-ark' # <--- CONFIRM YOUR GEE PROJECT ID
exported_agc_asset_id = f'projects/{my_gee_project_id}/assets/gaias_ark_agc_density_kenya'

# Load the exported AGC density image
agc_density_final = ee.Image(exported_agc_asset_id)
print(f"Loaded AGC Density asset: {exported_agc_asset_id}")

# --- Load the GMW Mangrove Extent (again, for consistency and ensuring it's available) ---
mangrove_collection = ee.ImageCollection("LANDSAT/MANGROVE_FORESTS")
recent_mangrove_image = mangrove_collection \
    .filterBounds(kenyan_coast_roi) \
    .sort('system:time_start', False) \
    .first()
if not recent_mangrove_image:
    raise Exception("No recent mangrove images found for the ROI in LANDSAT/MANGROVE_FORESTS collection.")

# The image is expected to be binary (1 for mangrove, 0 for non-mangrove).
# We rename the band to 'mangrove_presence' for clarity.
mangrove_presence_image = recent_mangrove_image.clip(kenyan_coast_roi).gt(0).unmask(0).rename('mangrove_presence')
print("Mangrove presence image loaded.")


# --- Create a consolidated Image for Carbon Analysis ---
# Stack the mangrove presence and AGC density into a single multi-band image.
# This creates one image where different bands represent different carbon-related metrics.
carbon_analysis_image = ee.Image.cat([
    mangrove_presence_image,
    agc_density_final
])
print("Consolidated image for carbon analysis created.")

# --- Optional: Visualize the consolidated image (AGC Density layer) ---
agc_vis_params = {
    'min': 0, 'max': 150,
    'palette': ['#ffffcc', '#a1dab4', '#41b6c4', '#2c7fb8', '#253494']
}

centroid_coords = kenyan_coast_roi.centroid().getInfo()['coordinates']
center_lat, center_lon = centroid_coords[1], centroid_coords[0]

m_consolidated_carbon = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles='OpenStreetMap')

folium.GeoJson(
    kenyan_coast_roi.getInfo(),
    name='Kenyan Coastal ROI',
    style_function=lambda x: {'fillColor': '#f08080', 'color': 'red', 'weight': 3, 'fillOpacity': 0.6}
).add_to(m_consolidated_carbon)

map_id_dict_agc = carbon_analysis_image.select('AGC_Density_tonnes_C_ha').getMapId(agc_vis_params)
folium.TileLayer(
    tiles=map_id_dict_agc['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='Consolidated AGC Density (tonnes C/ha)'
).add_to(m_consolidated_carbon)

folium.LayerControl().add_to(m_consolidated_carbon)
m_consolidated_carbon

# --- Optional: Export this consolidated image to GEE Assets ---
output_consolidated_asset_id = f'projects/{my_gee_project_id}/assets/gaias_ark_carbon_analysis_image_kenya'

# REVISED description: Simpler, no special characters like '&', shorter.
output_consolidated_description = 'Consolidated Mangrove Extent AGC Density Kenya' # <--- SIMPLIFIED DESCRIPTION

task_consolidated = ee.batch.Export.image.toAsset(
    image=carbon_analysis_image,
    description=output_consolidated_description,
    assetId=output_consolidated_asset_id,
    scale=30,
    region=kenyan_coast_roi.bounds(),
    maxPixels=1e10
)
task_consolidated.start()
print(f"\nGEE Export Task initiated for Consolidated Carbon Analysis Image. Asset ID: {output_consolidated_asset_id}")
print("Check the 'Tasks' tab in your GEE Code Editor to monitor progress.")

--- Consolidating Data for Carbon Analysis ---
Loaded AGC Density asset: projects/gaias-ark/assets/gaias_ark_agc_density_kenya
Mangrove presence image loaded.
Consolidated image for carbon analysis created.

GEE Export Task initiated for Consolidated Carbon Analysis Image. Asset ID: projects/gaias-ark/assets/gaias_ark_carbon_analysis_image_kenya
Check the 'Tasks' tab in your GEE Code Editor to monitor progress.
